# Uniformat Classification and Door Detection Training Notebook

This notebook fine-tunes YOLOv8 models for:
1. Uniformat level 3 and level 4 building element classification
2. Door detection

Adapted from the `train_models.py` script for Google Colab execution.

## 1. Setup Google Colab Environment

First, let's install the required packages and configure the Colab environment.

In [ ]:
# Install required packages
!pip install ultralytics
!pip install opencv-python matplotlib

In [ ]:
# Check GPU availability
!nvidia-smi

# Import required libraries
import os
import json
import time
from pathlib import Path
from google.colab import drive
from ultralytics import YOLO

## 2. Mount Google Drive

Mount your Google Drive to access datasets and save trained models.

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Define the base directory for the project in Google Drive
drive_base_dir = '/content/drive/MyDrive/UniformatCodesHorizon'
os.makedirs(drive_base_dir, exist_ok=True)
print(f"Project directory: {drive_base_dir}")

## 3. Upload Dataset Files

Upload your dataset YAML files to Google Colab.

In [ ]:
from google.colab import files

print("Upload your YOLOv8 format data.yaml files:")
uploaded = files.upload()

# Save uploaded files to Google Drive
dataset_dir = f"{drive_base_dir}/datasets"
os.makedirs(dataset_dir, exist_ok=True)

for filename in uploaded.keys():
    with open(f"{dataset_dir}/{filename}", 'wb') as f:
        f.write(uploaded[filename])
    print(f"Saved {filename} to {dataset_dir}/{filename}")

# List uploaded files
print("\nUploaded dataset files:")
!ls -la {dataset_dir}

## 4. Upload Dataset Images (Optional)

If your data.yaml files refer to local paths, you'll need to upload your dataset images as well.
For large datasets, it's better to upload them to Google Drive directly.

In [ ]:
# Uncomment and run this cell if you need to upload dataset images
'''
print("Upload your dataset zip file(s):")
uploaded = files.upload()

# Extract uploaded zip files
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        extract_path = f"{dataset_dir}/{filename.split('.')[0]}"
        os.makedirs(extract_path, exist_ok=True)
        !unzip -q -o {filename} -d {extract_path}
        print(f"Extracted {filename} to {extract_path}")
'''

## 5. Define Training Functions

In [ ]:
def train_model(
    data_yaml: str,
    output_dir: str,
    model_name: str,
    base_model: str = "yolov8n.pt",
    epochs: int = 10,
    batch_size: int = 16,
    img_size: int = 640,
    device: str = ""
) -> str:
    """
    Train a YOLOv8 model.
    
    Args:
        data_yaml: Path to the data.yaml file
        output_dir: Directory to save the trained model
        model_name: Name for the model output directory
        base_model: Base YOLOv8 model to fine-tune
        epochs: Number of training epochs
        batch_size: Batch size for training
        img_size: Image size for training
        device: Device to use for training
        
    Returns:
        Path to the trained model weights
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Load model
    model = YOLO(base_model)
    
    # Start training
    print(f"Starting training with data: {data_yaml}")
    print(f"Using base model: {base_model}")
    print(f"Training for {epochs} epochs with batch size {batch_size}")
    
    # Train the model
    model.train(
        data=data_yaml,
        epochs=epochs,
        batch=batch_size,
        imgsz=img_size,
        patience=20,  # Early stopping patience
        device=device,
        project=output_dir,
        name=model_name,
        exist_ok=True,
        pretrained=True,
        verbose=True
    )
    
    # Get the path to the best model weights
    best_weights_path = Path(output_dir) / model_name / "weights" / "best.pt"
    
    print(f"Training complete. Best weights saved to: {best_weights_path}")
    
    return str(best_weights_path)

## 6. Configure Training Parameters

Set up the parameters for model training.

In [ ]:
# Define training parameters
# Replace these paths with your actual data.yaml file paths after uploading
level3_data = f"{dataset_dir}/level3_data.yaml"  # Update with your actual filename
level4_data = f"{dataset_dir}/level4_data.yaml"  # Update with your actual filename
door_data = f"{dataset_dir}/door_data.yaml"      # Update with your actual filename

models_dir = f"{drive_base_dir}/models"
base_model = "yolov8n.pt"  # Options: yolov8n.pt, yolov8s.pt, yolov8m.pt, yolov8l.pt, yolov8x.pt
epochs = 10
batch_size = 16
img_size = 640
device = ""  # Auto-select (will use GPU in Colab)

# Create models output directory
os.makedirs(models_dir, exist_ok=True)

# Initialize training info dictionary
training_info = {
    "date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "base_model": base_model,
    "epochs": epochs,
    "batch_size": batch_size,
    "img_size": img_size,
}

## 7. Train Door Detection Model

Run this cell to train the door detection model.

In [ ]:
# Uncomment and run this cell to train the door detection model
'''
# Check if door data file exists
if os.path.exists(door_data):
    print("\n===== Training Door Detection Model =====\n")
    door_model_path = train_model(
        door_data,
        models_dir,
        "door_detection",
        base_model,
        epochs,
        batch_size,
        img_size,
        device
    )
    training_info["door_model"] = door_model_path
    training_info["door_data"] = door_data
else:
    print(f"Door dataset file not found at: {door_data}")
'''

## 8. Train Uniformat Level 3 Model

Run this cell to train the Uniformat Level 3 classification model.

In [ ]:
# Uncomment and run this cell to train the Uniformat Level 3 model
'''
# Check if level 3 data file exists
if os.path.exists(level3_data):
    print("\n===== Training Uniformat Level 3 Model =====\n")
    level3_model_path = train_model(
        level3_data,
        models_dir,
        "uniformat_level3",
        base_model,
        epochs,
        batch_size,
        img_size,
        device
    )
    training_info["level3_model"] = level3_model_path
    training_info["level3_data"] = level3_data
else:
    print(f"Level 3 dataset file not found at: {level3_data}")
'''

## 9. Train Uniformat Level 4 Model

Run this cell to train the Uniformat Level 4 classification model.

In [ ]:
# Uncomment and run this cell to train the Uniformat Level 4 model
'''
# Check if level 4 data file exists
if os.path.exists(level4_data):
    print("\n===== Training Uniformat Level 4 Model =====\n")
    level4_model_path = train_model(
        level4_data,
        models_dir,
        "uniformat_level4",
        base_model,
        epochs,
        batch_size,
        img_size,
        device
    )
    training_info["level4_model"] = level4_model_path
    training_info["level4_data"] = level4_data
else:
    print(f"Level 4 dataset file not found at: {level4_data}")
'''

## 10. Save Training Information

Save information about the training runs to a JSON file in Google Drive.

In [ ]:
# Save training info to Google Drive
training_info_path = f"{models_dir}/training_info.json"
with open(training_info_path, "w") as f:
    json.dump(training_info, f, indent=2)

print("\n===== Training Complete =====")
print(f"Training info saved to: {training_info_path}")

## 11. Evaluate Models

Evaluate the trained models on validation datasets.

In [ ]:
# Uncomment and run this cell to evaluate the models
'''
# Function to evaluate a model
def evaluate_model(model_path, data_yaml):
    if os.path.exists(model_path) and os.path.exists(data_yaml):
        print(f"\nEvaluating model: {model_path}")
        model = YOLO(model_path)
        results = model.val(data=data_yaml)
        print(f"Validation results: {results}")
    else:
        print(f"Model or data file not found: {model_path}, {data_yaml}")

# Evaluate trained models
if "door_model" in training_info:
    evaluate_model(training_info["door_model"], door_data)

if "level3_model" in training_info:
    evaluate_model(training_info["level3_model"], level3_data)
    
if "level4_model" in training_info:
    evaluate_model(training_info["level4_model"], level4_data)
'''

## 12. Run Inference Examples

Test the trained models with sample images.

In [ ]:
# Uncomment and run this cell to perform inference with trained models
'''
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# Upload test images
print("Upload test images:")
uploaded = files.upload()

# Function to run inference and display results
def run_inference(model_path, image_path):
    if os.path.exists(model_path) and os.path.exists(image_path):
        print(f"\nRunning inference on {image_path} with model {model_path}")
        
        # Load model
        model = YOLO(model_path)
        
        # Run inference
        results = model(image_path)
        
        # Display results
        for result in results:
            im_array = result.plot()  # plot a BGR numpy array of predictions
            im = cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB)  # convert to RGB
            plt.figure(figsize=(10, 10))
            plt.imshow(im)
            plt.axis('off')
            plt.show()
            
            # Print detection results
            print("\nDetection Results:")
            for i, box in enumerate(result.boxes):
                class_id = int(box.cls.item())
                class_name = result.names[class_id]
                conf = box.conf.item()
                print(f"  {i+1}. Class: {class_name}, Confidence: {conf:.2f}")
    else:
        print(f"Model or image file not found: {model_path}, {image_path}")

# Run inference on uploaded images with all trained models
for image_path in uploaded.keys():
    if "door_model" in training_info:
        run_inference(training_info["door_model"], image_path)
    
    if "level3_model" in training_info:
        run_inference(training_info["level3_model"], image_path)
        
    if "level4_model" in training_info:
        run_inference(training_info["level4_model"], image_path)
'''

## 13. Download Trained Models

Download the trained models to your local machine.

In [ ]:
# Modify the script
script_modifications = """
# Example modification: Add a print statement
print("Training script is running with modifications!")
"""

# Save modifications to the script
with open(script_path, "a") as file:
    file.write(script_modifications)

# Re-run the modified script
!python train_models.py